In [31]:
!pkill -f ngrok
!pkill -f streamlit

In [32]:
!pip -q install streamlit pyngrok bcrypt PyJWT

In [33]:
%%writefile app.py
import os
import re
import time
import jwt
import bcrypt
import secrets
import sqlite3
import smtplib
import datetime
import streamlit as st
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

DB_NAME = "freightiq_auth.db"
JWT_SECRET = os.getenv("JWT_SECRET", "change-this-secret-in-colab-secrets")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
ADMIN_USERNAME = "admin@portal.com"
ADMIN_PASSWORD = "AdminPassword@2026"
OTP_EXPIRY_SECONDS = 300

SECURITY_QUESTIONS = [
    "What is your favourite city?",
    "What is your pet name?",
    "What was your first school?",
    "What is your favourite shipment lane?"
]

st.set_page_config(
    page_title="FreightIQ Auth Console",
    page_icon="🚚",
    layout="wide",
    initial_sidebar_state="expanded"
)

def get_connection():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def hash_text(value):
    return bcrypt.hashpw(value.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")

def check_hash(value, hashed):
    try:
        return bcrypt.checkpw(value.encode("utf-8"), hashed.encode("utf-8"))
    except Exception:
        return False

def init_db():
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("""
            CREATE TABLE IF NOT EXISTS users (
                email TEXT PRIMARY KEY,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL,
                recent_login TEXT
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS password_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                password_hash TEXT NOT NULL,
                set_at TEXT NOT NULL
            )
        """)
        conn.commit()

    seed_users = [
        ("springboardmentor018@gmail.com", "Mentor_018", "Welcome@123", "What is your favourite city?", "bengaluru"),
        ("springboardmentor038@gmail.com", "Mentor_038", "Welcome@123", "What is your pet name?", "shadow"),
    ]
    for email, username, password, question, answer in seed_users:
        if not get_user_by_email(email):
            register_user(email, username, password, question, answer)

def valid_email(email):
    return re.match(r"^[A-Za-z]{2,}[\w.+-]*@[A-Za-z]{2,}\.[A-Za-z]{2,}$", email.strip()) is not None

def password_error(password):
    if len(password) < 8:
        return "Password must be at least 8 characters long."
    if not re.search(r"[A-Z]", password):
        return "Password must contain at least one uppercase letter."
    if not re.search(r"[a-z]", password):
        return "Password must contain at least one lowercase letter."
    if not re.search(r"\d", password):
        return "Password must contain at least one number."
    if not re.search(r"[^A-Za-z0-9]", password):
        return "Password must contain at least one special symbol."
    return ""

def password_strength(password):
    score = 0
    score += len(password) >= 8
    score += bool(re.search(r"[A-Z]", password) and re.search(r"[a-z]", password))
    score += bool(re.search(r"\d", password) and re.search(r"[^A-Za-z0-9]", password))
    return int(score)

def register_user(email, username, password, question, answer):
    now = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    password_hash = hash_text(password)
    answer_hash = hash_text(answer.lower().strip())
    with get_connection() as conn:
        c = conn.cursor()
        try:
            c.execute("""
                INSERT INTO users
                (email, username, password_hash, security_question, security_answer_hash, created_at, recent_login)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (email.lower().strip(), username.strip(), password_hash, question, answer_hash, now, "Never"))
            c.execute("INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)", (email.lower().strip(), password_hash, now))
            conn.commit()
            return True
        except sqlite3.IntegrityError:
            return False

def get_user_by_email(email):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT email, username, password_hash, security_question, security_answer_hash, created_at, recent_login FROM users WHERE email = ?", (email.lower().strip(),))
        return c.fetchone()

def get_user_by_username(username):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT email, username, password_hash, security_question, security_answer_hash, created_at, recent_login FROM users WHERE LOWER(username) = ?", (username.lower().strip(),))
        return c.fetchone()

def get_user_by_identity(identity):
    if "@" in identity:
        return get_user_by_email(identity)
    return get_user_by_username(identity)

def update_recent_login(email):
    value = datetime.datetime.now().strftime("%d %b %Y, %I:%M %p")
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("UPDATE users SET recent_login = ? WHERE email = ?", (value, email))
        conn.commit()
    return value

def authenticate(identity, password):
    user = get_user_by_identity(identity)
    if user and check_hash(password, user[2]):
        update_recent_login(user[0])
        return user
    return None

def make_token(email, username):
    payload = {
        "email": email,
        "username": username,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_token(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

def password_was_used(email, password):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT password_hash FROM password_history WHERE email = ?", (email.lower().strip(),))
        return any(check_hash(password, row[0]) for row in c.fetchall())

def update_password(email, new_password):
    now = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    password_hash = hash_text(new_password)
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("UPDATE users SET password_hash = ? WHERE email = ?", (password_hash, email.lower().strip()))
        c.execute("INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)", (email.lower().strip(), password_hash, now))
        conn.commit()

def get_all_users():
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT username, email, created_at, recent_login FROM users ORDER BY created_at DESC")
        return c.fetchall()

def send_otp_email(email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False
    msg = MIMEMultipart()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = email
    msg["Subject"] = "FreightIQ Password Reset OTP"
    msg.attach(MIMEText(f"Your FreightIQ OTP is {otp}. It expires in 5 minutes.", "plain"))
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, email, msg.as_string())
    return True

def load_css():
    st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800&display=swap');
    :root {
        --primary:#2563EB; --secondary:#06B6D4; --success:#22C55E;
        --background:#F8FAFC; --dark:#0F172A; --muted:#64748B;
    }
    html, body, [class*="css"] { font-family:'Poppins', sans-serif; }
    .stApp {
        background:
        radial-gradient(circle at top left, rgba(37,99,235,.18), transparent 35rem),
        radial-gradient(circle at bottom right, rgba(6,182,212,.20), transparent 35rem),
        var(--background);
    }
    [data-testid="stSidebar"] {
        background: linear-gradient(180deg, rgba(15,23,42,.96), rgba(30,41,59,.94));
    }
    [data-testid="stSidebar"] * { color: white !important; }
    .glass {
        padding: 28px;
        border: 1px solid rgba(148,163,184,.28);
        border-radius: 18px;
        background: rgba(255,255,255,.76);
        box-shadow: 0 24px 70px rgba(15,23,42,.14);
        backdrop-filter: blur(20px);
    }
    .hero {
        min-height: 560px;
        display: flex;
        align-items: center;
        padding: 50px 10px;
    }
    .hero-title {
        font-size: clamp(44px, 7vw, 88px);
        line-height: .96;
        font-weight: 800;
        color: #0F172A;
        margin-bottom: 18px;
    }
    .hero-sub {
        color: #475569;
        font-size: 18px;
        line-height: 1.7;
        max-width: 720px;
    }
    .eyebrow {
        display:inline-flex;
        gap:8px;
        align-items:center;
        color:#2563EB;
        font-weight:800;
        text-transform:uppercase;
        letter-spacing:.08em;
        font-size:13px;
    }
    .metric-grid {
        display:grid;
        grid-template-columns:repeat(3,1fr);
        gap:14px;
        margin-top:28px;
    }
    .metric, .feature, .dash-card {
        border:1px solid rgba(148,163,184,.28);
        border-radius:16px;
        padding:20px;
        background:rgba(255,255,255,.72);
        box-shadow:0 16px 40px rgba(15,23,42,.10);
    }
    .metric strong { color:#2563EB; font-size:24px; display:block; }
    .feature-grid, .dash-grid {
        display:grid;
        grid-template-columns:repeat(3,1fr);
        gap:18px;
        margin-top:24px;
    }
    .freight-art {
        min-height:360px;
        border-radius:24px;
        color:white;
        padding:28px;
        background:linear-gradient(135deg,#2563EB,#06B6D4);
        box-shadow:0 28px 80px rgba(37,99,235,.35);
    }
    .route {
        margin-top:65px;
        display:flex;
        align-items:center;
        justify-content:space-between;
        font-size:44px;
    }
    .route span {
        height:4px;
        flex:1;
        margin:0 16px;
        background:rgba(255,255,255,.65);
        border-radius:999px;
    }
    .status-pill {
        display:inline-block;
        padding:6px 12px;
        border-radius:999px;
        background:rgba(34,197,94,.16);
        color:#15803D;
        font-weight:700;
    }
    .danger-note {
        padding:14px 16px;
        border-radius:14px;
        background:rgba(239,68,68,.10);
        color:#991B1B;
        border:1px solid rgba(239,68,68,.18);
    }
    .footer {
        margin-top:50px;
        color:#64748B;
        border-top:1px solid rgba(148,163,184,.28);
        padding-top:20px;
    }
    .stButton > button {
        border-radius:12px;
        min-height:48px;
        font-weight:700;
        border:0;
        background:linear-gradient(135deg,#2563EB,#06B6D4);
        color:white;
        box-shadow:0 16px 35px rgba(37,99,235,.25);
    }
    .stButton > button:hover {
        transform:translateY(-2px);
        color:white;
    }
    @media (max-width:900px) {
        .metric-grid, .feature-grid, .dash-grid { grid-template-columns:1fr; }
        .hero-title { font-size:44px; }
    }
    </style>
    """, unsafe_allow_html=True)

def landing_page():
    left, right = st.columns([1.15, .85], gap="large")
    with left:
        st.markdown("""
        <div class="hero">
          <div>
            <div class="eyebrow">🔐 Enterprise freight authentication</div>
            <div class="hero-title">Intelligent Freight Quote Generation System</div>
            <div class="hero-sub">
              Secure access for logistics teams, quote analysts, and administrators managing modern freight quote workflows with JWT sessions, OTP recovery, and admin visibility.
            </div>
            <div class="metric-grid">
              <div class="metric"><strong>JWT</strong><span>Session ready</span></div>
              <div class="metric"><strong>OTP</strong><span>Email recovery</span></div>
              <div class="metric"><strong>Admin</strong><span>User visibility</span></div>
            </div>
          </div>
        </div>
        """, unsafe_allow_html=True)
        c1, c2 = st.columns(2)
        if c1.button("Login", use_container_width=True):
            st.session_state.page = "Login"
            st.rerun()
        if c2.button("Signup", use_container_width=True):
            st.session_state.page = "Signup"
            st.rerun()
    with right:
        st.markdown("""
        <div class="freight-art">
          <h2>Freight Route Quote</h2>
          <p>Origin: Bengaluru<br>Destination: Mumbai<br>Mode: Express<br>Estimated Quote: $2,840</p>
          <div class="route">🏭<span></span>🚚<span></span>📍</div>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("""
    <div class="feature-grid">
      <div class="feature"><h3>🔑 Secure Authentication</h3><p>Mandatory validation, password strength, and JWT session handling.</p></div>
      <div class="feature"><h3>📧 Dual Recovery</h3><p>Reset using either security question or email OTP.</p></div>
      <div class="feature"><h3>👥 Admin Dashboard</h3><p>View users without exposing passwords or hashes.</p></div>
    </div>
    <div class="footer">FreightIQ Authentication Console · Built for intelligent freight quote generation</div>
    """, unsafe_allow_html=True)

def login_page():
    st.markdown('<div class="glass">', unsafe_allow_html=True)
    st.subheader("Welcome back")
    st.caption("Sign in to continue to your freight quote workspace.")
    with st.form("login_form"):
        identity = st.text_input("Username/Email")
        password = st.text_input("Password", type="password")
        remember = st.checkbox("Remember Me")
        submitted = st.form_submit_button("Login", use_container_width=True)
    if submitted:
        if not identity or not password:
            st.error("All fields are mandatory.")
        else:
            user = authenticate(identity, password)
            if user:
                st.session_state.user = {"email": user[0], "username": user[1]}
                st.session_state.jwt_token = make_token(user[0], user[1])
                st.success("Login successful. JWT session is active.")
                time.sleep(1)
                st.session_state.page = "User Dashboard"
                st.rerun()
            else:
                st.error("Invalid username/email or password.")
    if st.button("Create Account"):
        st.session_state.page = "Signup"
        st.rerun()
    if st.button("Forgot Password"):
        st.session_state.page = "Forgot Password"
        st.rerun()
    st.markdown('</div>', unsafe_allow_html=True)

def signup_page():
    st.markdown('<div class="glass">', unsafe_allow_html=True)
    st.subheader("Create account")
    with st.form("signup_form"):
        c1, c2 = st.columns(2)
        username = c1.text_input("Username")
        email = c2.text_input("Email")
        password = c1.text_input("Password", type="password")
        confirm = c2.text_input("Confirm Password", type="password")
        question = st.selectbox("Security Question", [""] + SECURITY_QUESTIONS)
        answer = st.text_input("Security Answer")
        score = password_strength(password)
        st.progress(score / 3 if score else 0)
        st.caption(["Password strength", "Weak", "Medium", "Strong"][score])
        submitted = st.form_submit_button("Signup", use_container_width=True)
    if submitted:
        if not all([username, email, password, confirm, question, answer]):
            st.error("All fields are mandatory.")
        elif not valid_email(email):
            st.error("Email must be shaped like ab@cd.ef.")
        elif password_error(password):
            st.error(password_error(password))
        elif password != confirm:
            st.error("Password and Confirm Password must match.")
        elif not register_user(email, username, password, question, answer):
            st.error("Username or email already exists.")
        else:
            st.success("Signup successful. Please login.")
            time.sleep(1)
            st.session_state.page = "Login"
            st.rerun()
    st.markdown('</div>', unsafe_allow_html=True)

def forgot_password_page():
    st.subheader("Reset your password")
    tab1, tab2 = st.tabs(["Method 1: Security Question", "Method 2: Email OTP"])

    with tab1:
        with st.form("security_reset"):
            username = st.text_input("Username")
            question = st.selectbox("Security Question", [""] + SECURITY_QUESTIONS)
            answer = st.text_input("Security Answer")
            new_password = st.text_input("New Password", type="password")
            confirm = st.text_input("Confirm Password", type="password")
            submitted = st.form_submit_button("Reset Password", use_container_width=True)
        if submitted:
            user = get_user_by_username(username)
            if not all([username, question, answer, new_password, confirm]):
                st.error("All fields are mandatory.")
            elif not user:
                st.error("User not found.")
            elif user[3] != question or not check_hash(answer.lower().strip(), user[4]):
                st.error("Security verification failed.")
            elif password_error(new_password):
                st.error(password_error(new_password))
            elif new_password != confirm:
                st.error("Passwords do not match.")
            elif password_was_used(user[0], new_password):
                st.error("New password cannot reuse an old password.")
            else:
                update_password(user[0], new_password)
                st.success("Password reset successful. Please login.")

    with tab2:
        email = st.text_input("Email", key="otp_email")
        if st.button("Send OTP", use_container_width=True):
            user = get_user_by_email(email)
            if not email or not valid_email(email):
                st.error("Enter a valid email.")
            elif not user:
                st.error("Registered email not found.")
            else:
                otp = str(secrets.randbelow(900000) + 100000)
                st.session_state.otp = otp
                st.session_state.otp_email_value = email.lower().strip()
                st.session_state.otp_expiry = time.time() + OTP_EXPIRY_SECONDS
                try:
                    sent = send_otp_email(email, otp)
                    if sent:
                        st.success("OTP sent to your email.")
                    else:
                        st.info(f"Demo OTP: {otp}")
                except Exception:
                    st.info(f"Demo OTP: {otp}")
        with st.form("otp_reset"):
            otp_value = st.text_input("OTP Verification")
            new_password = st.text_input("New Password", type="password", key="otp_new")
            confirm = st.text_input("Confirm Password", type="password", key="otp_confirm")
            submitted = st.form_submit_button("Verify OTP and Reset", use_container_width=True)
        if submitted:
            email_for_reset = st.session_state.get("otp_email_value", "")
            if time.time() > st.session_state.get("otp_expiry", 0):
                st.error("OTP expired.")
            elif otp_value != st.session_state.get("otp", ""):
                st.error("OTP verification failed.")
            elif password_error(new_password):
                st.error(password_error(new_password))
            elif new_password != confirm:
                st.error("Passwords do not match.")
            elif password_was_used(email_for_reset, new_password):
                st.error("New password cannot reuse an old password.")
            else:
                update_password(email_for_reset, new_password)
                st.success("Password reset successful. Please login.")

def user_dashboard():
    token_data = verify_token(st.session_state.get("jwt_token", ""))
    if not token_data:
        st.error("Session expired. Please login again.")
        st.session_state.page = "Login"
        st.stop()
    user = get_user_by_email(token_data["email"])
    st.markdown(f"""
    <div class="glass">
      <div class="eyebrow">📊 User dashboard</div>
      <h1>Welcome, {user[1]}</h1>
      <p>{user[0]}</p>
    </div>
    <div class="dash-grid">
      <div class="dash-card"><h3>JWT Session Status</h3><span class="status-pill">Active and valid</span><p>Token issued for this user session.</p></div>
      <div class="dash-card"><h3>Recent Login</h3><p>{user[6]}</p></div>
      <div class="dash-card"><h3>Notification</h3><p>Quote generation workspace is secured and ready.</p></div>
    </div>
    """, unsafe_allow_html=True)
    if st.button("Logout", use_container_width=True):
        st.session_state.pop("jwt_token", None)
        st.session_state.pop("user", None)
        st.session_state.page = "Login"
        st.rerun()

def admin_login():
    st.markdown('<div class="glass">', unsafe_allow_html=True)
    st.subheader("Admin Login")
    with st.form("admin_login_form"):
        username = st.text_input("Admin Username")
        password = st.text_input("Admin Password", type="password")
        submitted = st.form_submit_button("Enter Admin Console", use_container_width=True)
    if submitted:
        if username.strip().lower() == ADMIN_USERNAME.lower() and password.strip() == ADMIN_PASSWORD:
            st.session_state.admin = True
            st.success("Admin console unlocked.")
            time.sleep(1)
            st.session_state.page = "Admin Dashboard"
            st.rerun()
        else:
            st.error("Invalid admin credentials.")
    st.markdown('</div>', unsafe_allow_html=True)

def admin_dashboard():
    if not st.session_state.get("admin"):
        st.error("Admin login required.")
        st.session_state.page = "Admin Login"
        st.stop()
    st.subheader("Registered Users")
    st.caption("Passwords and password hashes are never displayed.")
    users = get_all_users()
    st.dataframe(
        [{"Username": u[0], "Email": u[1], "Created At": u[2], "Recent Login": u[3]} for u in users],
        use_container_width=True,
        hide_index=True
    )

init_db()
load_css()

if "page" not in st.session_state:
    st.session_state.page = "Landing Page"

with st.sidebar:
    st.markdown("## 🚚 FreightIQ")
    st.caption("Quote Generation Console")
    pages = ["Landing Page", "Login", "Signup", "Forgot Password", "User Dashboard", "Admin Login", "Admin Dashboard"]
    selected = st.radio("Navigation", pages, index=pages.index(st.session_state.page))
    if selected != st.session_state.page:
        st.session_state.page = selected
        st.rerun()
    st.divider()
    dark = st.toggle("Dark Mode")
    if dark:
        st.markdown("""
        <style>
        .stApp { background: radial-gradient(circle at top left, rgba(37,99,235,.18), transparent 35rem), #0F172A; color:white; }
        .glass, .feature, .metric, .dash-card { background:rgba(15,23,42,.76); color:#E2E8F0; }
        .hero-title { color:#F8FAFC; }
        .hero-sub, p, .footer { color:#CBD5E1; }
        </style>
        """, unsafe_allow_html=True)

page = st.session_state.page
if page == "Landing Page":
    landing_page()
elif page == "Login":
    login_page()
elif page == "Signup":
    signup_page()
elif page == "Forgot Password":
    forgot_password_page()
elif page == "User Dashboard":
    user_dashboard()
elif page == "Admin Login":
    admin_login()
elif page == "Admin Dashboard":
    admin_dashboard()

Overwriting app.py


In [34]:
from pyngrok import ngrok
from google.colab import userdata
import os, time

ngrok_token = userdata.get("NGROK_AUTHTOKEN")
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print("Add NGROK_AUTHTOKEN in Colab Secrets first.")

get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &")
time.sleep(3)
public_url = ngrok.connect(8501)
print("Open your FreightIQ app here:", public_url)

Open your FreightIQ app here: NgrokTunnel: "https://retold-distort-magnolia.ngrok-free.dev" -> "http://localhost:8501"


In [15]:
!pkill ngrok
!pkill streamlit

In [16]:
from pyngrok import ngrok

# Disconnect and kill all existing active tunnels
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
    ngrok.disconnect(tunnel.public_url)

# Proceed with starting your new connection loop below